In [11]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset, random_split
from torch.nn.utils.rnn import pad_sequence
import time
import copy
import os
from pathlib import Path
import glob
import pandas as pd
import numpy as np
import timeit
import time
import sklearn
import yaml
from torchmetrics.classification import MulticlassF1Score
import csv

In [3]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"
device = get_device()
device

'cpu'

In [4]:
class SleepDataset(Dataset):
    def __init__(self, file_path, length, window_size=5):
        
        self.file_path = file_path
        self.length = length
        self.data = torch.load(self.file_path, weights_only=False, mmap=True)
#         self.features = torch.from_numpy(np.transpose(data["X"], axes=(0, 2, 1))).float()
#         self.labels = torch.from_numpy(data["y"]).long()


    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        
        # return self.features[idx], self.labels[idx]
        feature = torch.from_numpy(self.data["X"][idx]).float()
        label = torch.tensor(self.data["y"][idx]).long()
        
        # Apply transpose here if needed based on original shape
        feature = feature.transpose(1, 0)
        return feature, label
        

# data_dir = 'anphy_sleep_data/patient_records/clean'
data_dir = 'anphy_sleep_data/patient_records/only_body'        
file_paths = glob.glob(os.path.join(data_dir, '*.pt'))



In [12]:
results_dir = 'results_files'        
results_paths = glob.glob(f"{results_dir}/**/*.csv", recursive=True)
results_paths

['results_files/CNN-LSTM_base_optim_noscheduler_body_only_seed_59/seed_59_test_loss.csv',
 'results_files/CNN-LSTM_base_optim_noscheduler_body_only_seed_59/seed_59_val_loss.csv',
 'results_files/CNN-LSTM_base_optim_noscheduler_body_only_seed_59/seed_59_train_loss.csv',
 'results_files/CNN_ADAM_optim_with_scheduler_seed_42/seed_42_train_loss.csv',
 'results_files/CNN_ADAM_optim_with_scheduler_seed_42/seed_42_test_loss.csv',
 'results_files/CNN_ADAM_optim_with_scheduler_seed_42/seed_42_val_loss.csv',
 'results_files/CNN-LSTM_ADAM_optim_with_scheduler_seed_42/seed_42_train_loss.csv',
 'results_files/CNN-LSTM_ADAM_optim_with_scheduler_seed_42/seed_42_test_loss.csv',
 'results_files/CNN-LSTM_ADAM_optim_with_scheduler_seed_42/seed_42_val_loss.csv',
 'results_files/CNN-LSTM_WITH_CONTEXT_base_optim_noscheduler_seed_42/seed_42_train_loss.csv',
 'results_files/CNN-LSTM_WITH_CONTEXT_base_optim_noscheduler_seed_42/seed_42_test_loss.csv',
 'results_files/CNN-LSTM_WITH_CONTEXT_base_optim_noscheduler

In [18]:
results_paths[0]

'results_files/CNN-LSTM_base_optim_noscheduler_body_only_seed_59/seed_59_test_loss.csv'

In [13]:
from pandas.api.types import is_string_dtype
for file in results_paths:
    df = pd.read_csv(file, index_col=[0])
    # print(df)
    if is_string_dtype(df["f1"]):
        df["f1"] = df['f1'].str.extract(r'tensor\(([\d.]+)').astype(float)
        df.to_csv(file, index_label="epoch")
        print(f"{file}: ", df.iloc[0, :])
    else:
        print(f"{file} is already formatted")



results_files/CNN-LSTM_base_optim_noscheduler_body_only_seed_59/seed_59_test_loss.csv is already formatted
results_files/CNN-LSTM_base_optim_noscheduler_body_only_seed_59/seed_59_val_loss.csv is already formatted
results_files/CNN-LSTM_base_optim_noscheduler_body_only_seed_59/seed_59_train_loss.csv is already formatted
results_files/CNN_ADAM_optim_with_scheduler_seed_42/seed_42_train_loss.csv is already formatted
results_files/CNN_ADAM_optim_with_scheduler_seed_42/seed_42_test_loss.csv is already formatted
results_files/CNN_ADAM_optim_with_scheduler_seed_42/seed_42_val_loss.csv is already formatted
results_files/CNN-LSTM_ADAM_optim_with_scheduler_seed_42/seed_42_train_loss.csv is already formatted
results_files/CNN-LSTM_ADAM_optim_with_scheduler_seed_42/seed_42_test_loss.csv is already formatted
results_files/CNN-LSTM_ADAM_optim_with_scheduler_seed_42/seed_42_val_loss.csv is already formatted
results_files/CNN-LSTM_WITH_CONTEXT_base_optim_noscheduler_seed_42/seed_42_train_loss.csv:  ac

In [8]:
pd.read_csv(results_paths[0], index_col=[0]).rename_axis('test')

,accuracy,loss,f1
test,,,
0,0.516713,1.24422,0.2886


In [14]:
for file in results_paths:
    df = pd.read_csv(file, index_col=[0])
    df = df.rename_axis('epoch')
    df.to_csv(file, index_label="epoch")
    
        

In [10]:
pd.read_csv(results_paths[0], index_col=[0])

,accuracy,loss,f1
epoch,,,
0,0.516713,1.24422,0.2886


In [ ]:
# 
# lengths = [torch.load(file_path, weights_only=False)["y"].shape[0] for file_path in file_paths]
# # with open('recording_epoch_nums.csv', 'w', newline='') as f:
# with open('recording_epoch_nums_body_only.csv', 'w', newline='') as f:
#     writer = csv.writer(f)
#     for i in range(len(file_paths)):
#         writer.writerow([file_paths[i].replace(f"{data_dir}/", ""), lengths[i]]) # writerow takes a list

In [6]:
# metadata = pd.read_csv("recording_epoch_nums.csv", header=None)
metadata = pd.read_csv("recording_epoch_nums_body_only.csv", header=None)
metadata.sort_values(by=0, inplace=True)
metadata = metadata.reset_index(drop=True)
metadata

,0,1
0,EPCTL01_body_dataset.pt,957
1,EPCTL02_body_dataset.pt,1072
2,EPCTL03_body_dataset.pt,843
3,EPCTL04_body_dataset.pt,762
4,EPCTL05_body_dataset.pt,755
5,EPCTL06_body_dataset.pt,919
6,EPCTL07_body_dataset.pt,979
7,EPCTL08_body_dataset.pt,873
8,EPCTL09_body_dataset.pt,945
9,EPCTL10_body_dataset.pt,950


In [ ]:
metadata.loc[:, 1].sum()

In [20]:

datasets = [SleepDataset(data_dir + "/" + metadata.loc[i, 0], metadata.loc[i, 1]) for i in range(len(metadata))]

In [21]:
combined_dataset = ConcatDataset(datasets)

In [22]:
combined_dataset[0][0].shape

torch.Size([3000, 14])

In [23]:
len(combined_dataset)

26118

In [10]:
def custom_collate(batch):
#     print(f"Length of batch: {len(batch)}")
#     print(f"First item in batch's first element: {batch[0][0].shape}")
#     print(f"First item in batch's second element: {batch[0][1].shape}")
    data = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    # Pad sequences to the length of the longest in the batch
#     print(f"Data list: {data}")
#     print(f"Labels list: {labels}" )
    padded_data = pad_sequence(data, batch_first=True)
    labels = torch.tensor(labels)
    return padded_data, labels
